In [1]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pickle
from copy import copy

parent_dir = os.path.abspath("..")
if parent_dir not in sys.path:
    print(f"appending {parent_dir} to path")
    sys.path.append(parent_dir)

import src.unit_conv as conv

k_B = 1.380649e-23
Mass = 164*1.660539*(1e-24/k_B)

appending /home/artur/inzyn/quantum_decaf/view to path


In [2]:
quantum_root = Path(parent_dir).parents[0]
fname = "./70x70x30small/all/time_evol_runs.pickle"
fname = quantum_root / fname

with open(fname, 'rb') as f:
    runs = pickle.load(f)

xr = [runs[0]["r_right"][0][0]]
xl = [runs[0]["r_left"][0][0]]
xrm = [runs[0]["r_right_max"][0][0]]
xlm = [runs[0]["r_left_max"][0][0]]

times = [0]
n_part = runs[0]["n_part"]

e_tot = [runs[0]["energies"]["total"][0]*conv.hartree_to_nK]
e_kin = [runs[0]["energies"]["kinetic"][0]*conv.hartree_to_nK]
e_ext = [runs[0]["energies"]["ext"][0]*conv.hartree_to_nK]
e_contact = [runs[0]["energies"]["contact"][0]*conv.hartree_to_nK]
e_contact_ext = [runs[0]["energies"]["contact_ext"][0]*conv.hartree_to_nK]
e_dd = [runs[0]["energies"]["dipole"][0]*conv.hartree_to_nK]

for i, run in enumerate(runs):
    r_right = run["r_right"][0][1:]
    r_left = run["r_left"][0][1:]
    rm_right = run["r_right_max"][0][1:]
    rm_left = run["r_left_max"][0][1:]

    tot = run["energies"]["total"][1:]*conv.hartree_to_nK
    kin = run["energies"]["kinetic"][1:]*conv.hartree_to_nK
    ext = run["energies"]["ext"][1:]*conv.hartree_to_nK
    contact = run["energies"]["contact"][1:]*conv.hartree_to_nK
    contact_ext = run["energies"]["contact_ext"][1:]*conv.hartree_to_nK
    dd = run["energies"]["dipole"][1:]*conv.hartree_to_nK

    n_steps = run["n_steps"]
    dt = run["timestep"]*conv.jat_to_ms
    timestamps = np.arange(1, n_steps + 1)*dt
    times.extend(timestamps + copy(times[-1]))

    xr.extend(r_right.copy())
    xl.extend(r_left.copy())
    xrm.extend(rm_right.copy())
    xlm.extend(rm_left.copy())

    e_tot.extend(tot.copy())
    e_kin.extend(kin.copy())
    e_ext.extend(ext.copy())
    e_contact.extend(contact.copy())
    e_contact_ext.extend(contact_ext.copy())
    e_dd.extend(dd.copy())

xr = np.array(xr)
xl = np.array(xl)
xrm = np.array(xrm)
xlm = np.array(xlm)

e_tot = np.array(e_tot)
e_kin = np.array(e_kin)
e_ext = np.array(e_ext)
e_contact = np.array(e_contact)
e_contact_ext = np.array(e_contact_ext)
e_dd = np.array(e_dd)

times = np.array(times)

In [3]:
%matplotlib qt
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(times, xr, "b")
ax.plot(times, xrm, "b--")
ax.plot(times, xl, "r")
ax.plot(times, xlm, "r--")
plt.show()

In [4]:
vr = np.empty_like(xr)
vl = np.empty_like(xl)

vr[0] = (xr[1] - xr[0])/dt
vr[-1] = (xr[-1] - xr[-2])/dt

vl[0] = (xl[1] - xl[0])/dt
vl[-1] = (xl[-1] - xl[-2])/dt

vr[1:-1] = 0.5*(xr[2:] - xr[0:-2])/dt
vl[1:-1] = 0.5*(xl[2:] - xl[0:-2])/dt

In [5]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(times, xr, label="xr(t)", color="black")
plt.legend()
ax2 = ax.twinx()
ax2.plot(times, vr, label="vr(t)", color="red")
plt.legend()
ax2.set_ylabel(r"v $[\mu m / ms]$")
ax.set_ylabel(r"x $[\mu m]$")
ax.set_xlabel("t [ms]")
plt.show()

In [6]:
tr = 0.5*(n_part*0.5)*Mass*vr**2
tl = 0.5*(n_part*0.5)*Mass*vl**2

fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(times, e_kin, label=r"$\hat T \psi$")
ax.plot(times, tr+tl, label="classic")

plt.legend()
plt.show()

/home/artur/inzyn/quantum_decaf/view/.venv/lib/python3.14/site-packages/matplotlib/cbook.py:1811: ComplexWarning: Casting complex values to real discards the imaginary part
  return math.isfinite(val)
/home/artur/inzyn/quantum_decaf/view/.venv/lib/python3.14/site-packages/matplotlib/cbook.py:1408: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asanyarray(x, float)


In [7]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(times, tr + tl, label="classic kin")
ax.plot(times, e_dd - e_dd[0], label=r"$\Delta$ dd(t)")
ax.plot(times, e_contact - e_contact[0], label=r"$\Delta$ cont(t)")
ax.plot(times, e_contact_ext - e_contact_ext[0], label=r"$\Delta$ cont_ext(t)")
ax.plot(times, e_ext - e_ext[0], label=r"$\Delta$ ext(t)")
ax.set_ylabel(r"E $[nK]$")
ax.set_xlabel("t [ms]")
plt.legend()
plt.show()

In [8]:
dd_eff = e_tot - e_ext - e_contact - e_contact_ext - tr - tl
dd_eff = dd_eff - dd_eff[0]
print(f"shape of eff is {dd_eff.shape}")
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(times, e_dd - e_dd[0], label="quantum_dd")
ax.plot(times, dd_eff, label="eff dd")
plt.legend()
plt.xlabel("t[ms]")
plt.ylabel("E[nK]")
plt.show()

shape of eff is (3001,)


In [9]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(times, e_kin - tr - tl, label="quantum T - classi T")
plt.xlabel("t [ms]")
plt.ylabel("E [nK]")
plt.legend()
plt.show()